# **MÓDULO 35 - Cross Validation**

Nesta tarefa, você trabalhará com uma base de dados que contém informações sobre variáveis ambientais coletadas para a detecção de incêndios. O objetivo é utilizar técnicas de validação cruzada (cross-validation) para avaliar a performance de um modelo de classificação na previsão da ocorrência de um incêndio com base nas variáveis fornecidas.


Descrição da Base de Dados
A base de dados contém as seguintes variáveis:

Unnamed:0: Índice (não é uma variável útil para o modelo)

UTC: Tempo em Segundos UTC

Temperature[C]: Temperatura do Ar (em graus Celsius)

Humidity[%]: Umidade do Ar (em porcentagem)

TVOC[ppb]: Total de Compostos Orgânicos Voláteis (medido em partes por bilhão)

eCO2[ppm]: Concentração equivalente de CO2 (medido em partes por milhão)

Raw H2: Hidrogênio molecular bruto, não compensado

Raw Ethanol: Etanol gasoso bruto

Pressure[hPA]: Pressão do Ar (em hectopascais)

PM1.0: Material particulado de tamanho < 1,0 µm

PM2.5: Material particulado de tamanho >1,0 µm e < 2,5 µm

NC0.5: Concentração numérica de material particulado de tamanho < 0,5 µm

NC1.0: Concentração numérica de material particulado de tamanho 0,5 µm < 1,0 µm

NC2.5: Concentração numérica de material particulado de tamanho 1,0 µm < 2,5 µm

CNT: Contador de amostras


E a variável alvo:

Fire Alarm: Indicador binário de incêndio (1 se houver incêndio, 0 caso contrário)

O objetivo desta tarefa é aplicar a técnica de validação cruzada (cross-validation) para avaliar a performance de um modelo de classificação. A validação cruzada ajudará a garantir que o modelo seja avaliado de maneira robusta e generalize bem para dados não vistos.

In [1]:
import pandas as pd
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score
from sklearn.ensemble import RandomForestClassifier

# 1 - Carregue a base de dados, verifique os tipos de dados e também se há presença de dados faltantes ou nulos.

In [2]:
# Carregar dados
df = pd.read_csv('Cientista de dados M35 - smoke_detection_iot.csv', index_col='Unnamed: 0')
df.rename(columns={'Fire Alarm': 'Fire_Alarm'}, inplace=True)

In [3]:
df

,UTC,Temperature[C],Humidity[%],TVOC[ppb],eCO2[ppm],Raw H2,Raw Ethanol,Pressure[hPa],PM1.0,PM2.5,NC0.5,NC1.0,NC2.5,CNT,Fire_Alarm
0,1654733331,20.000,57.36,0,400,12306,18520,939.735,0.00,0.00,0.00,0.000,0.000,0,0
1,1654733332,20.015,56.67,0,400,12345,18651,939.744,0.00,0.00,0.00,0.000,0.000,1,0
2,1654733333,20.029,55.96,0,400,12374,18764,939.738,0.00,0.00,0.00,0.000,0.000,2,0
3,1654733334,20.044,55.28,0,400,12390,18849,939.736,0.00,0.00,0.00,0.000,0.000,3,0
4,1654733335,20.059,54.69,0,400,12403,18921,939.744,0.00,0.00,0.00,0.000,0.000,4,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
62625,1655130047,18.438,15.79,625,400,13723,20569,936.670,0.63,0.65,4.32,0.673,0.015,5739,0
62626,1655130048,18.653,15.87,612,400,13731,20588,936.678,0.61,0.63,4.18,0.652,0.015,5740,0
62627,1655130049,18.867,15.84,627,400,13725,20582,936.687,0.57,0.60,3.95,0.617,0.014,5741,0
62628,1655130050,19.083,16.04,638,400,13712,20566,936.680,0.57,0.59,3.92,0.611,0.014,5742,0


In [4]:
# Análise dos tipos de dados do dataframe
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 62630 entries, 0 to 62629
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   UTC             62630 non-null  int64  
 1   Temperature[C]  62630 non-null  float64
 2   Humidity[%]     62630 non-null  float64
 3   TVOC[ppb]       62630 non-null  int64  
 4   eCO2[ppm]       62630 non-null  int64  
 5   Raw H2          62630 non-null  int64  
 6   Raw Ethanol     62630 non-null  int64  
 7   Pressure[hPa]   62630 non-null  float64
 8   PM1.0           62630 non-null  float64
 9   PM2.5           62630 non-null  float64
 10  NC0.5           62630 non-null  float64
 11  NC1.0           62630 non-null  float64
 12  NC2.5           62630 non-null  float64
 13  CNT             62630 non-null  int64  
 14  Fire_Alarm      62630 non-null  int64  
dtypes: float64(8), int64(7)
memory usage: 7.6 MB


Observa-se que todas as variáveis do conjunto de dados são numéricas, sendo do tipo `int` ou `float`, o que é ideal para a aplicação de modelos de Machine Learning.

Além disso, nota-se a ausência de valores nulos no dataframe. Todas as colunas possuem 62.630 registros não nulos, quantidade que corresponde exatamente ao número total de linhas do conjunto de dados.

# 2 - Para essa base, onde você realizará as previsões de fire alarm, qual modelo de machine learning você aplicará? Justifique.




Como o objetivo desta atividade é classificar a ocorrência de incêndios, foi escolhido o modelo Random Forest. Além de ser amplamente utilizado em problemas de classificação, esse algoritmo se destaca por sua robustez e capacidade de lidar com relações complexas nos dados. Dessa forma, espera-se que o modelo apresente um bom desempenho e resultados satisfatórios na predição de incêndios.

# 3 - Separe a base em Y e X e já rode a instância do modelo que você utilizará.

In [5]:
# Separação em variáveis dependentes e independetes
y = df['Fire_Alarm']
X = df.drop('Fire_Alarm', axis=1)

In [6]:
# Iniciando o modelo Random Forest
rf_model = RandomForestClassifier(random_state=37)

# 4 - Defina o número de Folds e rode o modelo com a validação cruzada.

In [7]:
# Número de folds
Kfold = 7

In [8]:
# Define a validação cruzada
crossvalidation = KFold(n_splits=Kfold, shuffle=True, random_state=37)

In [9]:
# Roda o modelo com a validação cruzada
modelo_final = cross_val_score(rf_model, X, y, cv = crossvalidation)

Foi utilizado o modelo Random Forest com validação cruzada, adotando um total de 7 folds para a avaliação do desempenho do modelo.

# 5 - Avalie a pontuação de cada modelo e ao final a validação final da média.

In [10]:
# Pontuação dos modelos
print(f"Pontuações por fold: {modelo_final}")

# Média das pontuações
print(f"Média dos modelos da Validação Cruzada: {modelo_final.mean()}")

Pontuações por fold: [1.         0.99988823 1.         1.         0.99988823 1.
 1.        ]
Média dos modelos da Validação Cruzada: 0.9999680659119577


Os resultados obtidos pelo modelo foram excepcionais. Em 5 dos 7 folds avaliados, o Random Forest alcançou pontuação perfeita. Nos outros dois folds, a acurácia obtida foi de 0.999888, um valor extremamente próximo de 1. Ao final da validação cruzada, a média das pontuações foi de 0.99996, indicando um desempenho praticamente perfeito do modelo.

## Portanto, com a Validação Cruzada, verificamos que o modelo apresentou um desempenho realmente excelente, não sendo apenas um resultado obtido por acaso devido a um subconjunto específico de treino e teste.